In [0]:
# Purpose  : Clean and standardize product data

In [0]:
silver_products = spark.table(
    "workspace.indian_ecommerce_sales_analytics.bronze_products"
)

display(silver_products)

Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp
PROD000001,Samsung Mobile V5,Electronics,Samsung,130561.34,23,30029.11,100532.23,159,2.63,4.49,102,products.csv,2026-09-10T06:30:10.616Z
PROD000002,HP Laptop V5,Electronics,HP,36494.92,34,12408.27,24086.65,54,4.19,4.32,107,products.csv,2026-09-10T06:30:10.616Z
PROD000003,boAt Headphones V2,Electronics,boAt,106587.94,42,44766.93,61821.01,638,1.48,4.31,99,products.csv,2026-09-10T06:30:10.616Z
PROD000004,Noise Watch V7,Electronics,Noise,81275.27,47,38199.38,43075.89,572,3.3,4.42,127,products.csv,2026-09-10T06:30:10.616Z
PROD000005,Apple iPhone V4,Electronics,Apple,13099.36,43,5632.72,7466.64,865,3.71,4.37,98,products.csv,2026-09-10T06:30:10.616Z
PROD000006,Levis Jeans V1,Fashion,Levis,3330.44,9,299.74,3030.7,380,2.64,4.47,81,products.csv,2026-09-10T06:30:10.616Z
PROD000007,Peter England Shirt V8,Fashion,Peter,7504.54,12,900.54,6604.0,800,2.12,4.37,82,products.csv,2026-09-10T06:30:10.616Z
PROD000008,Nike Shoes V2,Fashion,Nike,7210.81,38,2740.11,4470.7,321,1.71,4.29,104,products.csv,2026-09-10T06:30:10.616Z
PROD000009,Titan Watch V10,Fashion,Titan,8545.82,48,4101.99,4443.83,266,0.82,4.52,94,products.csv,2026-09-10T06:30:10.616Z
PROD000010,Puma T-shirt V7,Fashion,Puma,12528.96,35,4385.14,8143.82,834,3.12,4.37,89,products.csv,2026-09-10T06:30:10.616Z


In [0]:
silver_products.printSchema()
print("Product Row Count:", silver_products.count())

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Original_Price: double (nullable = true)
 |-- Discount_Percent: integer (nullable = true)
 |-- Discount_Amount: double (nullable = true)
 |-- Selling_Price: double (nullable = true)
 |-- Stock_Quantity: integer (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- Avg_Rating: double (nullable = true)
 |-- Total_Reviews: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)

Product Row Count: 2000


In [0]:
#remove unnecessary spaces from text fields
from pyspark.sql import functions as F
string_columns = [
    "Product_ID",
    "Product_Name",
    "Category",
    "Brand"
]

for col_name in string_columns:
    silver_products = silver_products.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

In [0]:
#Standardize Category
silver_products = silver_products.withColumn(
    "Category",
    F.initcap(F.lower(F.trim(F.col("Category"))))
)
display(
    silver_products
    .groupBy("Category")
    .count()
    .orderBy(F.desc("count"))
)

Category,count
Electronics,290
Books,285
Beauty,285
Home,285
Fashion,285
Sports,285
Grocery,285


In [0]:
#Standartize brand names
silver_products = silver_products.withColumn(
    "Brand",
    F.initcap(F.lower(F.trim(F.col("Brand"))))
)
display(
    silver_products
    .groupBy("Brand")
    .count()
    .orderBy(F.desc("count"))
)

Brand,count
Puma,114
Nike,114
Boat,58
Hp,58
Samsung,58
Noise,58
Apple,58
Aashirvaad,57
Ncert,57
Fortune,57


In [0]:
silver_products = (
    silver_products
    .withColumn("Original_Price", F.col("Original_Price").cast("double"))
    .withColumn("Discount_Percent", F.col("Discount_Percent").cast("double"))
    .withColumn("Discount_Amount", F.col("Discount_Amount").cast("double"))
    .withColumn("Selling_Price", F.col("Selling_Price").cast("double"))
    .withColumn("Stock_Quantity", F.col("Stock_Quantity").cast("int"))
    .withColumn("Weight_kg", F.col("Weight_kg").cast("double"))
    .withColumn("Avg_Rating", F.col("Avg_Rating").cast("double"))
    .withColumn("Total_Reviews", F.col("Total_Reviews").cast("int"))
)

In [0]:
#Duplicate product IDs
duplicate_products = (
    silver_products
    .groupBy("Product_ID")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate Product IDs:",
    duplicate_products.count()
)

display(duplicate_products)

Duplicate Product IDs: 0


Product_ID,count


In [0]:
#Negative prices
invalid_prices = silver_products.filter(
    (F.col("Original_Price") < 0) |
    (F.col("Discount_Amount") < 0) |
    (F.col("Selling_Price") < 0)
)

print(
    "Invalid Price Records:",
    invalid_prices.count()
)

display(invalid_prices)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6389001320841693>, line 2
      1 #Negative prices
----> 2 invalid_prices = silver_products.filter(
      3     (F.col("Original_Price") < 0) |
      4     (F.col("Discount_Amount") < 0) |
      5     (F.col("Selling_Price") < 0)
      6 )
      8 print(
      9     "Invalid Price Records:",
     10     invalid_prices.count()
     11 )
     13 display(invalid_prices)

NameError: name 'silver_products' is not defined

In [0]:
#Invalid Stocks
invalid_stock = silver_products.filter(
    F.col("Stock_Quantity") < 0
)

print(
    "Invalid Stock Records:",
    invalid_stock.count()
)

display(invalid_stock)

Invalid Stock Records: 0


Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp


In [0]:
#Invalid Ratings
invalid_rating = silver_products.filter(
    (F.col("Avg_Rating") < 0) |
    (F.col("Avg_Rating") > 5)
)

print(
    "Invalid Rating Records:",
    invalid_rating.count()
)

display(invalid_rating)


Invalid Rating Records: 0


Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp


In [0]:
#Invalid reviews
invalid_reviews = silver_products.filter(
    F.col("Total_Reviews") < 0
)

print(
    "Invalid Review Records:",
    invalid_reviews.count()
)

display(invalid_reviews)

Invalid Review Records: 0


Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp


In [0]:
#Checking price logic
price_logic_issue = silver_products.filter(
    F.col("Selling_Price") > F.col("Original_Price")
)

print(
    "Selling Price > Original Price:",
    price_logic_issue.count()
)

display(price_logic_issue)

Selling Price > Original Price: 0


Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp


In [0]:
#Adding Data Quality Flags
silver_products = (
    silver_products

    .withColumn(
        "dq_invalid_price",
        (
            (F.col("Original_Price") < 0) |
            (F.col("Discount_Amount") < 0) |
            (F.col("Selling_Price") < 0)
        )
    )

    .withColumn(
        "dq_invalid_discount",
        (
            (F.col("Discount_Percent") < 0) |
            (F.col("Discount_Percent") > 100)
        )
    )

    .withColumn(
        "dq_invalid_stock",
        F.col("Stock_Quantity") < 0
    )

    .withColumn(
        "dq_invalid_rating",
        (
            (F.col("Avg_Rating") < 0) |
            (F.col("Avg_Rating") > 5)
        )
    )

    .withColumn(
        "dq_invalid_reviews",
        F.col("Total_Reviews") < 0
    )

    .withColumn(
        "dq_price_logic_issue",
        F.col("Selling_Price") > F.col("Original_Price")
    )
)

In [0]:
#Null Values checking
null_report = silver_products.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in silver_products.columns
])

display(null_report)


Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,_source_file,_ingestion_timestamp,dq_invalid_price,dq_invalid_discount,dq_invalid_stock,dq_invalid_rating,dq_invalid_reviews,dq_price_logic_issue
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
silver_products = silver_products.select(
    "Product_ID",
    "Product_Name",
    "Category",
    "Brand",
    "Original_Price",
    "Discount_Percent",
    "Discount_Amount",
    "Selling_Price",
    "Stock_Quantity",
    "Weight_kg",
    "Avg_Rating",
    "Total_Reviews",
    "dq_invalid_price",
    "dq_invalid_discount",
    "dq_invalid_stock",
    "dq_invalid_rating",
    "dq_invalid_reviews",
    "dq_price_logic_issue"
)

In [0]:
silver_products.printSchema()

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Original_Price: double (nullable = true)
 |-- Discount_Percent: double (nullable = true)
 |-- Discount_Amount: double (nullable = true)
 |-- Selling_Price: double (nullable = true)
 |-- Stock_Quantity: integer (nullable = true)
 |-- Weight_kg: double (nullable = true)
 |-- Avg_Rating: double (nullable = true)
 |-- Total_Reviews: integer (nullable = true)
 |-- dq_invalid_price: boolean (nullable = true)
 |-- dq_invalid_discount: boolean (nullable = true)
 |-- dq_invalid_stock: boolean (nullable = true)
 |-- dq_invalid_rating: boolean (nullable = true)
 |-- dq_invalid_reviews: boolean (nullable = true)
 |-- dq_price_logic_issue: boolean (nullable = true)



In [0]:
print(
    "Final Silver Product Count:",
    silver_products.count()
)

print(
    "Distinct Product IDs:",
    silver_products
    .select("Product_ID")
    .distinct()
    .count()
)

Final Silver Product Count: 2000
Distinct Product IDs: 2000


In [0]:
display(
    silver_products.limit(20)
)

Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,dq_invalid_price,dq_invalid_discount,dq_invalid_stock,dq_invalid_rating,dq_invalid_reviews,dq_price_logic_issue
PROD000001,Samsung Mobile V5,Electronics,Samsung,130561.34,23.0,30029.11,100532.23,159,2.63,4.49,102,false,false,false,false,false,false
PROD000002,HP Laptop V5,Electronics,Hp,36494.92,34.0,12408.27,24086.65,54,4.19,4.32,107,false,false,false,false,false,false
PROD000003,boAt Headphones V2,Electronics,Boat,106587.94,42.0,44766.93,61821.01,638,1.48,4.31,99,false,false,false,false,false,false
PROD000004,Noise Watch V7,Electronics,Noise,81275.27,47.0,38199.38,43075.89,572,3.3,4.42,127,false,false,false,false,false,false
PROD000005,Apple iPhone V4,Electronics,Apple,13099.36,43.0,5632.72,7466.64,865,3.71,4.37,98,false,false,false,false,false,false
PROD000006,Levis Jeans V1,Fashion,Levis,3330.44,9.0,299.74,3030.7,380,2.64,4.47,81,false,false,false,false,false,false
PROD000007,Peter England Shirt V8,Fashion,Peter,7504.54,12.0,900.54,6604.0,800,2.12,4.37,82,false,false,false,false,false,false
PROD000008,Nike Shoes V2,Fashion,Nike,7210.81,38.0,2740.11,4470.7,321,1.71,4.29,104,false,false,false,false,false,false
PROD000009,Titan Watch V10,Fashion,Titan,8545.82,48.0,4101.99,4443.83,266,0.82,4.52,94,false,false,false,false,false,false
PROD000010,Puma T-shirt V7,Fashion,Puma,12528.96,35.0,4385.14,8143.82,834,3.12,4.37,89,false,false,false,false,false,false


In [0]:
#Saving silver products data to delta
(
    silver_products.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.silver_products"
    )
)

In [0]:
%sql
SELECT COUNT(*) AS total_products
FROM workspace.indian_ecommerce_sales_analytics.silver_products;

total_products
2000


In [0]:
%sql
SELECT *
FROM workspace.indian_ecommerce_sales_analytics.silver_products
LIMIT 30;

Product_ID,Product_Name,Category,Brand,Original_Price,Discount_Percent,Discount_Amount,Selling_Price,Stock_Quantity,Weight_kg,Avg_Rating,Total_Reviews,dq_invalid_price,dq_invalid_discount,dq_invalid_stock,dq_invalid_rating,dq_invalid_reviews,dq_price_logic_issue
PROD000001,Samsung Mobile V5,Electronics,Samsung,130561.34,23.0,30029.11,100532.23,159,2.63,4.49,102,false,false,false,false,false,false
PROD000002,HP Laptop V5,Electronics,Hp,36494.92,34.0,12408.27,24086.65,54,4.19,4.32,107,false,false,false,false,false,false
PROD000003,boAt Headphones V2,Electronics,Boat,106587.94,42.0,44766.93,61821.01,638,1.48,4.31,99,false,false,false,false,false,false
PROD000004,Noise Watch V7,Electronics,Noise,81275.27,47.0,38199.38,43075.89,572,3.3,4.42,127,false,false,false,false,false,false
PROD000005,Apple iPhone V4,Electronics,Apple,13099.36,43.0,5632.72,7466.64,865,3.71,4.37,98,false,false,false,false,false,false
PROD000006,Levis Jeans V1,Fashion,Levis,3330.44,9.0,299.74,3030.7,380,2.64,4.47,81,false,false,false,false,false,false
PROD000007,Peter England Shirt V8,Fashion,Peter,7504.54,12.0,900.54,6604.0,800,2.12,4.37,82,false,false,false,false,false,false
PROD000008,Nike Shoes V2,Fashion,Nike,7210.81,38.0,2740.11,4470.7,321,1.71,4.29,104,false,false,false,false,false,false
PROD000009,Titan Watch V10,Fashion,Titan,8545.82,48.0,4101.99,4443.83,266,0.82,4.52,94,false,false,false,false,false,false
PROD000010,Puma T-shirt V7,Fashion,Puma,12528.96,35.0,4385.14,8143.82,834,3.12,4.37,89,false,false,false,false,false,false
